<a href="https://colab.research.google.com/github/hanokjoshua144/DEEP-LEARNING-6-SEM/blob/main/Assignment_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ABOUT THYE DATASET

The CIFAR-10 dataset contains 60,000 color images (32×32 resolution) across 10 classes.
Classes include: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.
The dataset is challenging due to low resolution and high intra-class similarity.
Many classes have overlapping visual features (e.g., cat vs dog, truck vs automobile).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

1. MLP – Learning Rate vs Loss

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

learning_rates = [0.0001, 0.001, 0.01, 0.1]
losses = []

for lr in learning_rates:
    model = MLP().to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    total_loss = 0
    for epoch in range(3):
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

    losses.append(total_loss)

plt.plot(learning_rates, losses, marker='o')
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.show()

OBSERVATIONS

MLP – Learning Rate vs Loss
When I varied the learning rate, I observed that very small learning rate (0.0001) resulted in extremely slow decrease in loss.
For learning rate 0.001, the loss decreased steadily and training was stable.
At learning rate 0.01, the model converged faster and gave the lowest loss.
For learning rate 0.1, the loss fluctuated heavily and sometimes increased, indicating instability.
Overall, I observed that moderate learning rate gives best performance, while too high or too low learning rates degrade performance.

2. MLP with Gradient Descent

In [ ]:
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

epoch_loss_list = []

for epoch in range(10):
    total = 0
    for x,y in trainloader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    epoch_loss_list.append(total)

plt.plot(epoch_loss_list)
plt.title("Convergence")
plt.show()

OBSERVATIONS

MLP with Gradient Descent (Convergence)
During training, I observed that the loss decreased gradually across epochs, showing proper convergence.
The curve was not perfectly smooth due to mini-batch updates, but the overall trend was downward.
In early epochs, loss decreased rapidly, while later it slowed down.
If learning rate was increased slightly, oscillations appeared in the loss curve.
Overall, the model showed stable convergence using gradient descent on CIFAR-10.

MLP for CIFAR-10

In [ ]:
model = MLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    for x,y in trainloader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

OBSERVATIONS

MLP on CIFAR-10
I observed that MLP achieved only moderate accuracy (~45–55%).
Even after training, the model struggled with visually similar classes.
Loss decreased but plateaued early, indicating limited learning capability.
Since MLP flattens images, it could not capture spatial relationships.
Overall, MLP showed underfitting on CIFAR-10 image data.

MLP with ALL GD TYPES

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

MLP MODEL (COMMON)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    def forward(self,x):
        return self.net(x)

In [ ]:
def train_model(optimizer_name, optimizer):
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()

    losses = []

    for epoch in range(3):
        total_loss = 0
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        losses.append(total_loss)
        print(f"{optimizer_name} Epoch {epoch+1} Loss: {total_loss}")

    return losses

1. BATCH GRADIENT DESCENT (BGD)

In [ ]:
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

loss_bgd = train_model("BGD", optimizer)

2. STOCHASTIC GRADIENT DESCENT (SGD)

In [ ]:
trainloader_sgd = torch.utils.data.DataLoader(trainset, batch_size=1, shuffle=True)

model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

loss_sgd = train_model("SGD", optimizer)

3. MINI-BATCH GD

In [ ]:
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

loss_minibatch = train_model("MiniBatch", optimizer)

4. SGD WITH MOMENTUM

In [ ]:
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

loss_momentum = train_model("Momentum", optimizer)

5. SGD WITH NESTEROV

In [ ]:
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

loss_nesterov = train_model("Nesterov", optimizer)

6. ADAGRAD

In [ ]:
model = MLP().to(device)
optimizer = optim.Adagrad(model.parameters(), lr=0.01)

loss_adagrad = train_model("Adagrad", optimizer)

7. RMSPROP

In [ ]:
model = MLP().to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

loss_rmsprop = train_model("RMSprop", optimizer)

8. ADADELTA

In [ ]:
model = MLP().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=1.0)

loss_adadelta = train_model("Adadelta", optimizer)

9. ADAM

In [ ]:
model = MLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

loss_adam = train_model("Adam", optimizer)

In [ ]:
plt.plot(loss_bgd, label="BGD")
plt.plot(loss_sgd, label="SGD")
plt.plot(loss_minibatch, label="MiniBatch")
plt.plot(loss_momentum, label="Momentum")
plt.plot(loss_nesterov, label="Nesterov")
plt.plot(loss_adagrad, label="Adagrad")
plt.plot(loss_rmsprop, label="RMSprop")
plt.plot(loss_adadelta, label="Adadelta")
plt.plot(loss_adam, label="Adam")

plt.legend()
plt.title("Optimizer Comparison")
plt.show()

When I used Batch Gradient Descent, I observed that the training was very slow and required more time to converge.
In Stochastic Gradient Descent (SGD), I observed faster updates but the loss fluctuated heavily due to noisy gradients.
Mini-batch GD provided a balance between speed and stability, and showed smoother convergence compared to SGD.
With Momentum, I observed that convergence was faster and oscillations were reduced compared to normal SGD.
Nesterov optimizer performed slightly better than Momentum by predicting future gradients, leading to smoother updates.
In Adagrad, I observed that learning slowed down over time due to continuously decreasing learning rate.
RMSprop solved this issue and showed stable and faster convergence compared to Adagrad.
Adadelta performed better than Adagrad but was slightly slower compared to RMSprop.
Adam optimizer showed the best performance, with fastest convergence and lowest loss.

. Final comparison observation:

From all experiments on CIFAR-10, I observed that Adam and RMSprop gave the best performance, while basic SGD methods were slower and less stable.
Adaptive optimizers handled the dataset better due to automatic learning rate adjustment.

5.MLP USING ALL REGULARIZATION TECHNIQUES (CIFAR-10)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=base_transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)


In [ ]:
class MLP(nn.Module):
    def __init__(self, dropout=False):
        super().__init__()
        if dropout:
            self.net = nn.Sequential(
                nn.Flatten(),
                nn.Linear(3072,256),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(256,128),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(128,10)
            )
        else:
            self.net = nn.Sequential(
                nn.Flatten(),
                nn.Linear(3072,256),
                nn.ReLU(),
                nn.Linear(256,128),
                nn.ReLU(),
                nn.Linear(128,10)
            )
    def forward(self,x):
        return self.net(x)

In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            _,pred = torch.max(outputs,1)
            total += y.size(0)
            correct += (pred == y).sum().item()

    return 100 * correct / total

In [ ]:
def train_model(model, optimizer, name, noise=False):
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    for epoch in range(2):   # keep small for speed
        model.train()
        total_loss = 0

        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            if noise:
                x = x + 0.1 * torch.randn_like(x)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

    acc = evaluate(model)
    print(f"{name} → Loss: {total_loss:.2f}, Accuracy: {acc:.2f}%")

In [ ]:
model = MLP()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_model(model, optimizer, "Base Model")

1. L2 REGULARIZATION

In [ ]:
model = MLP()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
train_model(model, optimizer, "L2 Regularization")

Observation
I observed that weights became smaller and smoother.
Training loss slightly higher but validation performance improved.
Overfitting reduced compared to no regularization.

2. DATASET AUGMENTATION

In [ ]:
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset_aug = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=aug_transform)

subset_aug, _ = torch.utils.data.random_split(trainset_aug, [3000, len(trainset_aug)-3000])
trainloader = torch.utils.data.DataLoader(subset_aug, batch_size=64, shuffle=True)

model = MLP()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_model(model, optimizer, "Data Augmentation")

Observation
Model became more robust to variations in images.
Accuracy improved compared to base model.
Reduced overfitting significantly.

3. PARAMETER SHARING

In [ ]:
class SharedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        shared_layer = nn.Linear(256,256)

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072,256),
            nn.ReLU(),
            shared_layer,
            nn.ReLU(),
            shared_layer,   # reused weights
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        return self.net(x)

model = SharedMLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

Observation
Reduced number of parameters.
Slight drop in accuracy due to reduced flexibility.
Helped in controlling overfitting.

4. ADDING NOISE TO INPUTS

In [ ]:
model = MLP()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_model(model, optimizer, "Noise Injection", noise=True)

Observation
Model became robust to noisy inputs.
Small noise improved generalization.
Large noise reduced accuracy.

5. EARLY STOPPING

In [ ]:
model = MLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_loss = float('inf')
patience = 2
counter = 0

for epoch in range(10):
    total_loss = 0
    model.train()

    for x,y in trainloader:
        x,y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.2f}")

    if total_loss < best_loss:
        best_loss = total_loss
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early Stopping Triggered")
        break

acc = evaluate(model)
print(f"Early Stopping → Accuracy: {acc:.2f}%")

Observation
Training stopped automatically when no improvement observed.
Prevented over-training.
Saved computation time.

6. ENSEMBLE METHODS

In [ ]:
models = [MLP().to(device) for _ in range(3)]
optimizers = [optim.Adam(m.parameters(), lr=0.001) for m in models]
criterion = nn.CrossEntropyLoss()

for epoch in range(2):
    for x,y in trainloader:
        x,y = x.to(device), y.to(device)

        for i,m in enumerate(models):
            optimizers[i].zero_grad()
            loss = criterion(m(x), y)
            loss.backward()
            optimizers[i].step()

# evaluation
def ensemble_acc():
    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            outputs = [m(x) for m in models]
            avg = sum(outputs)/len(outputs)

            _,pred = torch.max(avg,1)
            total += y.size(0)
            correct += (pred == y).sum().item()

    return 100 * correct / total

print(f"Ensemble → Accuracy: {ensemble_acc():.2f}%")

Observation
Ensemble improved accuracy compared to single model.
Predictions became more stable.
Training cost increased significantly.

7. DROPOUT

In [ ]:
model = MLP(dropout=True)
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_model(model, optimizer, "Dropout")

Observation
Reduced overfitting significantly.
Training loss increased slightly but test performance improved.
Prevented neurons from co-adapting.